In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import tensorflow as tf
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"GPU Device: {tf.test.gpu_device_name()}")

if not tf.config.list_physical_devices('GPU'):
    print("\nWARNING: GPU not detected!")
else:
    print("\nGPU enabled and ready")

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU Device: /device:GPU:0

GPU enabled and ready


In [4]:
!pip install vaderSentiment
import numpy as np
import pandas as pd
import joblib
import warnings
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [6]:
!pip install optuna
import optuna

In [7]:
import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.2.0


In [8]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

In [9]:
warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

In [10]:
# Set project path
project_path = '/content/drive/MyDrive/sentiment_analysis_project'
print(f"Project path: {project_path}")

Project path: /content/drive/MyDrive/sentiment_analysis_project


In [11]:
# STEP 2: Load all feature matrices from Drive
print("Loading feature matrices from Drive...")

# Load features
X_train_tfidf = joblib.load(f'{project_path}/features/X_train_tfidf.pkl')
X_val_tfidf = joblib.load(f'{project_path}/features/X_val_tfidf.pkl')

X_train_w2v = joblib.load(f'{project_path}/features/X_train_w2v.pkl')
X_val_w2v = joblib.load(f'{project_path}/features/X_val_w2v.pkl')

X_train_use = joblib.load(f'{project_path}/features/X_train_use.pkl')
X_val_use = joblib.load(f'{project_path}/features/X_val_use.pkl')

X_train_bert = joblib.load(f'{project_path}/features/X_train_bert.pkl')
X_val_bert = joblib.load(f'{project_path}/features/X_val_bert.pkl')

# Load labels
y_train = joblib.load(f'{project_path}/features/y_train.pkl')
y_val = joblib.load(f'{project_path}/features/y_val.pkl')

print("\nAll features loaded successfully")

Loading feature matrices from Drive...

All features loaded successfully


In [12]:
# Verify shapes and create summary
print("\n" + "="*80)
print("LOADED FEATURES SUMMARY")
print("="*80)

summary_data = {
    'Feature Set': ['TF-IDF', 'Word2Vec', 'USE', 'BERT', 'Labels'],
    'Train Shape': [
        X_train_tfidf.shape,
        X_train_w2v.shape,
        X_train_use.shape,
        X_train_bert.shape,
        y_train.shape
    ],
    'Val Shape': [
        X_val_tfidf.shape,
        X_val_w2v.shape,
        X_val_use.shape,
        X_val_bert.shape,
        y_val.shape
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("="*80)


LOADED FEATURES SUMMARY
Feature Set   Train Shape     Val Shape
     TF-IDF (75149, 5000) (18788, 5000)
   Word2Vec  (75149, 300)  (18788, 300)
        USE  (75149, 512)  (18788, 512)
       BERT  (75149, 384)  (18788, 384)
     Labels      (75149,)      (18788,)


## Train VADER Sentiment Analyzer

In [13]:
# We need the original text for VADER,
df_full = pd.read_csv('/content/drive/MyDrive/sentiment_analysis_project/yelp_train_100k.csv')

In [16]:
# Debug: check the actual split
print(f"X_val_text length: {len(X_val_text)}")
print(f"y_val length: {len(y_val)}")
print(f"vader_preds length: {len(vader_preds)}")

NameError: name 'X_val_text' is not defined

In [17]:
# STEP 3: VADER Baseline (FINAL FIX)
print("="*80)
print("BASELINE MODEL 1: VADER")
print("="*80)

# Load the CSV
df_full = pd.read_csv('/content/drive/MyDrive/sentiment_analysis_project/yelp_train_100k.csv')
print(f"Original CSV: {len(df_full)} rows")

# Sort by text length and remove shortest
df_full['text_length'] = df_full['text'].fillna('').astype(str).str.len()
df_full = df_full.sort_values('text_length').iloc[6063:].reset_index(drop=True)
df_full = df_full.drop('text_length', axis=1)

print(f"After removing shortest 6,063: {len(df_full)} rows")

# Now split
from sklearn.model_selection import train_test_split
X_train_text, X_val_text, y_train_split, y_val_split = train_test_split(
    df_full['text'],
    df_full['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_full['label']
)

print(f"\nValidation samples: {len(X_val_text)}")
print(f"Expected (y_val): {len(y_val)}")
print(f"Match: {len(X_val_text) == len(y_val)}")

# Initialize VADER
vader = SentimentIntensityAnalyzer()

# Run VADER
print("\nRunning VADER sentiment analysis...")
vader_scores = [vader.polarity_scores(text)['compound'] for text in X_val_text]
vader_preds = (np.array(vader_scores) >= 0).astype(int)

# Metrics
vader_acc = accuracy_score(y_val, vader_preds)
vader_f1 = f1_score(y_val, vader_preds)

print(f"\nVADER Results:")
print(f"  Accuracy: {vader_acc:.4f}")
print(f"  F1 Score: {vader_f1:.4f}")

BASELINE MODEL 1: VADER
Original CSV: 100000 rows
After removing shortest 6,063: 93937 rows

Validation samples: 18788
Expected (y_val): 18788
Match: True

Running VADER sentiment analysis...

VADER Results:
  Accuracy: 0.5009
  F1 Score: 0.6063


## TextBlob

In [18]:
# STEP 4: TextBlob Baseline
print("\n" + "="*80)
print("BASELINE MODEL 2: TEXTBLOB")
print("="*80)

# Run TextBlob on same validation text
print("Running TextBlob sentiment analysis...")
textblob_scores = [TextBlob(text).sentiment.polarity for text in X_val_text]
textblob_preds = (np.array(textblob_scores) >= 0).astype(int)

# Metrics
textblob_acc = accuracy_score(y_val, textblob_preds)
textblob_f1 = f1_score(y_val, textblob_preds)

print(f"\nTextBlob Results:")
print(f"  Accuracy: {textblob_acc:.4f}")
print(f"  F1 Score: {textblob_f1:.4f}")


BASELINE MODEL 2: TEXTBLOB
Running TextBlob sentiment analysis...

TextBlob Results:
  Accuracy: 0.4988
  F1 Score: 0.6120


In [19]:
# Baseline comparison table
print("\n" + "="*80)
print("BASELINE MODELS COMPARISON")
print("="*80)

baseline_results = {
    'Model': ['VADER', 'TextBlob'],
    'Accuracy': [f"{vader_acc:.4f}", f"{textblob_acc:.4f}"],
    'F1 Score': [f"{vader_f1:.4f}", f"{textblob_f1:.4f}"],
    'Method': ['Lexicon-based', 'Lexicon + patterns']
}

baseline_df = pd.DataFrame(baseline_results)
print(baseline_df.to_string(index=False))


BASELINE MODELS COMPARISON
   Model Accuracy F1 Score             Method
   VADER   0.5009   0.6063      Lexicon-based
TextBlob   0.4988   0.6120 Lexicon + patterns


## Logistic Regression with 5-Fold Cross-Validation

In [20]:
# STEP 5: Logistic Regression with 5-fold CV
print("\n" + "="*80)
print("MODEL 1: LOGISTIC REGRESSION (TF-IDF)")
print("="*80)

# Initialize model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

# 5-fold stratified cross-validation on TRAINING set
print("Running 5-fold stratified cross-validation...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Get CV scores
cv_scores_acc = cross_val_score(lr_model, X_train_tfidf, y_train, cv=skf, scoring='accuracy', n_jobs=-1)
cv_scores_f1 = cross_val_score(lr_model, X_train_tfidf, y_train, cv=skf, scoring='f1', n_jobs=-1)

print(f"\n5-Fold CV Results:")
print(f"  Accuracy: {cv_scores_acc.mean():.4f} (+/- {cv_scores_acc.std():.4f})")
print(f"  F1 Score: {cv_scores_f1.mean():.4f} (+/- {cv_scores_f1.std():.4f})")

# Train on full training set
print("\nTraining final model on full training set...")
lr_model.fit(X_train_tfidf, y_train)

# Evaluate on validation set
y_pred_lr = lr_model.predict(X_val_tfidf)
lr_acc = accuracy_score(y_val, y_pred_lr)
lr_f1 = f1_score(y_val, y_pred_lr)

print(f"\nValidation Set Results:")
print(f"  Accuracy: {lr_acc:.4f}")
print(f"  F1 Score: {lr_f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_val, y_pred_lr, target_names=['Negative', 'Positive']))


MODEL 1: LOGISTIC REGRESSION (TF-IDF)
Running 5-fold stratified cross-validation...

5-Fold CV Results:
  Accuracy: 0.9177 (+/- 0.0013)
  F1 Score: 0.9178 (+/- 0.0012)

Training final model on full training set...

Validation Set Results:
  Accuracy: 0.9200
  F1 Score: 0.9200

Classification Report:
              precision    recall  f1-score   support

    Negative       0.92      0.92      0.92      9389
    Positive       0.92      0.92      0.92      9399

    accuracy                           0.92     18788
   macro avg       0.92      0.92      0.92     18788
weighted avg       0.92      0.92      0.92     18788



In [21]:
# STEP 6: Save model and initialize results tracker
import os

# Create models directory if it doesn't exist
os.makedirs(f'{project_path}/models', exist_ok=True)

# Save Logistic Regression model
joblib.dump(lr_model, f'{project_path}/models/logistic_regression.pkl')
print(f"\nModel saved: logistic_regression.pkl")

# Initialize results tracker (will add all models here)
results_tracker = {
    'VADER': {'accuracy': vader_acc, 'f1': vader_f1, 'type': 'Baseline'},
    'TextBlob': {'accuracy': textblob_acc, 'f1': textblob_f1, 'type': 'Baseline'},
    'Logistic Regression': {'accuracy': lr_acc, 'f1': lr_f1, 'type': 'Classical ML'}
}

print("\n" + "="*80)
print("RESULTS SO FAR")
print("="*80)
for model_name, scores in results_tracker.items():
    print(f"{model_name:25s} | Acc: {scores['accuracy']:.4f} | F1: {scores['f1']:.4f} | {scores['type']}")
print("="*80)


Model saved: logistic_regression.pkl

RESULTS SO FAR
VADER                     | Acc: 0.5009 | F1: 0.6063 | Baseline
TextBlob                  | Acc: 0.4988 | F1: 0.6120 | Baseline
Logistic Regression       | Acc: 0.9200 | F1: 0.9200 | Classical ML


## Support Vector Machine (SVM)

In [22]:
# STEP 7: Linear SVM (FAST)
print("\n" + "="*80)
print("MODEL 2: LINEAR SVM (TF-IDF)")
print("="*80)

from sklearn.svm import LinearSVC

# Linear SVM is MUCH faster than RBF kernel
svm_model = LinearSVC(
    C=1.0,
    max_iter=1000,
    random_state=42
)

print("Training Linear SVM...")
svm_model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred_svm = svm_model.predict(X_val_tfidf)
svm_acc = accuracy_score(y_val, y_pred_svm)
svm_f1 = f1_score(y_val, y_pred_svm)

print(f"\nValidation Set Results:")
print(f"  Accuracy: {svm_acc:.4f}")
print(f"  F1 Score: {svm_f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_val, y_pred_svm, target_names=['Negative', 'Positive']))


MODEL 2: LINEAR SVM (TF-IDF)
Training Linear SVM...

Validation Set Results:
  Accuracy: 0.9171
  F1 Score: 0.9172

Classification Report:
              precision    recall  f1-score   support

    Negative       0.92      0.92      0.92      9389
    Positive       0.92      0.92      0.92      9399

    accuracy                           0.92     18788
   macro avg       0.92      0.92      0.92     18788
weighted avg       0.92      0.92      0.92     18788



## XGBoost

In [23]:
# STEP 9: XGBoost (NO TUNING - FAST)
print("\n" + "="*80)
print("MODEL 3: XGBOOST (TF-IDF)")
print("="*80)

# Use solid default parameters (no tuning needed)
xgb_model = xgb.XGBClassifier(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)

print("Training XGBoost with default parameters...")
xgb_model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred_xgb = xgb_model.predict(X_val_tfidf)
xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)

print(f"\nValidation Set Results:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  F1 Score: {xgb_f1:.4f}")

print(f"\nClassification Report:")
print(classification_report(y_val, y_pred_xgb, target_names=['Negative', 'Positive']))


MODEL 3: XGBOOST (TF-IDF)
Training XGBoost with default parameters...

Validation Set Results:
  Accuracy: 0.8924
  F1 Score: 0.8920

Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.90      0.89      9389
    Positive       0.90      0.89      0.89      9399

    accuracy                           0.89     18788
   macro avg       0.89      0.89      0.89     18788
weighted avg       0.89      0.89      0.89     18788



In [24]:
# STEP 10: Save XGBoost
joblib.dump(xgb_model, f'{project_path}/models/xgboost.pkl')

results_tracker['XGBoost'] = {'accuracy': xgb_acc, 'f1': xgb_f1, 'type': 'Classical ML'}

print("\n" + "="*80)
print("CLASSICAL ML MODELS COMPLETE")
print("="*80)
for model_name, scores in results_tracker.items():
    print(f"{model_name:25s} | Acc: {scores['accuracy']:.4f} | F1: {scores['f1']:.4f}")
print("="*80)


CLASSICAL ML MODELS COMPLETE
VADER                     | Acc: 0.5009 | F1: 0.6063
TextBlob                  | Acc: 0.4988 | F1: 0.6120
Logistic Regression       | Acc: 0.9200 | F1: 0.9200
XGBoost                   | Acc: 0.8924 | F1: 0.8920


## DISTILBERT FINE-TUNING (PyTorch)

In [2]:
# Check what's installed
import transformers
print(f"Transformers version: {transformers.__version__}")

# Install TensorFlow support for transformers
!pip install transformers[tf]

Transformers version: 5.2.0


In [4]:
# Alternative: Use PyTorch version (more compatible)
!pip install transformers torch

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.10.0+cu128
CUDA available: True


In [9]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset


In [10]:
# STEP 11: Load DistilBERT and prepare data (PyTorch)
print("\n" + "="*80)
print("MODEL 4: DISTILBERT FINE-TUNING (PyTorch)")
print("="*80)

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

# Load tokenizer and model
print("Loading DistilBERT tokenizer and model...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

# Move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Load text data
df_full = pd.read_csv('/content/drive/MyDrive/sentiment_analysis_project/yelp_train_100k.csv')
df_full['text_length'] = df_full['text'].fillna('').astype(str).str.len()
df_full = df_full.sort_values('text_length').iloc[6063:].reset_index(drop=True)
df_full = df_full.drop('text_length', axis=1)

X_train_text, X_val_text, y_train_text, y_val_text = train_test_split(
    df_full['text'],
    df_full['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_full['label']
)

print(f"Text data: {len(X_train_text)} train, {len(X_val_text)} val")


MODEL 4: DISTILBERT FINE-TUNING (PyTorch)
Loading DistilBERT tokenizer and model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
Parameters: 66,955,010
Text data: 75149 train, 18788 val


In [11]:
# Tokenize data
print("\nTokenizing data...")
train_encodings = tokenizer(
    X_train_text.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    X_val_text.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete")


Tokenizing data...
Tokenization complete


In [13]:
# Create PyTorch Dataset
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SentimentDataset(train_encodings, y_train_text.values)
val_dataset = SentimentDataset(val_encodings, y_val_text.values)

print(f"Datasets created: {len(train_dataset)} train, {len(val_dataset)} val")

Datasets created: 75149 train, 18788 val


In [15]:
# Complete setup for DistilBERT training
from google.colab import drive
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Mount drive
drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/sentiment_analysis_project'

# Load results tracker
results_tracker = {
    'VADER': {'accuracy': 0.5021, 'f1': 0.6059, 'type': 'Baseline'},
    'TextBlob': {'accuracy': 0.5018, 'f1': 0.6141, 'type': 'Baseline'},
    'Logistic Regression': {'accuracy': 0.9200, 'f1': 0.9200, 'type': 'Classical ML'},
    'Linear SVM': {'accuracy': 0.9171, 'f1': 0.9172, 'type': 'Classical ML'},
    'XGBoost': {'accuracy': 0.8924, 'f1': 0.8920, 'type': 'Classical ML'}
}

print("Setup complete - ready for DistilBERT training")
print(f"GPU available: {torch.cuda.is_available()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete - ready for DistilBERT training
GPU available: True


In [17]:
# STEP 12: Fine-tune with Trainer API (FIXED)
print("\n" + "="*80)
print("FINE-TUNING DISTILBERT (3 epochs)")
print("="*80)

# Training arguments (updated for newer transformers)
training_args = TrainingArguments(
    output_dir=f'{project_path}/models/distilbert_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=f'{project_path}/logs',
    logging_steps=500,
    eval_strategy='epoch',  # Changed from evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    fp16=True,
    report_to='none'
)

# Define metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train
print("\nStarting training...")

trainer.train()

print("\nTraining complete!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



FINE-TUNING DISTILBERT (3 epochs)

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.205930,0.187213,0.931552,0.929750
2,0.134069,0.229598,0.934905,0.934152
3,0.057543,0.300869,0.936821,0.936138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training complete!


In [18]:
# Evaluate and save
print("\nEvaluating on validation set...")
eval_results = trainer.evaluate()

print(f"\nDistilBERT Results:")
print(f"  Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_results['eval_f1']:.4f}")

# Get detailed predictions
predictions = trainer.predict(val_dataset)
y_pred_bert = predictions.predictions.argmax(-1)

print(f"\nClassification Report:")
print(classification_report(y_val_text, y_pred_bert, target_names=['Negative', 'Positive']))

# Save model
model.save_pretrained(f'{project_path}/models/distilbert_final')
tokenizer.save_pretrained(f'{project_path}/models/distilbert_final')
print(f"\nModel saved: distilbert_final/")

# Update tracker
bert_acc = eval_results['eval_accuracy']
bert_f1 = eval_results['eval_f1']
results_tracker['DistilBERT'] = {'accuracy': bert_acc, 'f1': bert_f1, 'type': 'Deep Learning'}

print("\n" + "="*80)
print("FINAL MODEL COMPARISON")
print("="*80)
sorted_models = sorted(results_tracker.items(), key=lambda x: x[1]['f1'], reverse=True)
for model_name, scores in sorted_models:
    print(f"{model_name:25s} | Acc: {scores['accuracy']:.4f} | F1: {scores['f1']:.4f} | {scores['type']}")
print("="*80)


Evaluating on validation set...



DistilBERT Results:
  Accuracy: 0.9368
  F1 Score: 0.9361

Classification Report:
              precision    recall  f1-score   support

    Negative       0.94      0.93      0.94      9549
    Positive       0.93      0.94      0.94      9239

    accuracy                           0.94     18788
   macro avg       0.94      0.94      0.94     18788
weighted avg       0.94      0.94      0.94     18788



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved: distilbert_final/

FINAL MODEL COMPARISON
DistilBERT                | Acc: 0.9368 | F1: 0.9361 | Deep Learning
Logistic Regression       | Acc: 0.9200 | F1: 0.9200 | Classical ML
Linear SVM                | Acc: 0.9171 | F1: 0.9172 | Classical ML
XGBoost                   | Acc: 0.8924 | F1: 0.8920 | Classical ML
TextBlob                  | Acc: 0.5018 | F1: 0.6141 | Baseline
VADER                     | Acc: 0.5021 | F1: 0.6059 | Baseline
